## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
)

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
RESULTS_DIR = "results/lasso_results" # Create this directory

## 2. Load Data

In [ ]:
df = pd.read_csv("data/cibil_score/cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

In [ ]:
df["approved_flag"].value_counts()

## 3. Feature / Target Setup
* `X` -> all columns except `credit_score` and `approved_flag`
* `y_linear` -> `credit_score` (Linear Regression target)
* `y_multiclass` -> `approved_flag` (Logistic Regression, multiclass: P1/P2/P3/P4)
* `y_binary` -> `approved_flag` mapped to **1** for P1/P2 and **0** for P3/P4 (Logistic Regression, binary)

In [ ]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

## 4. Train / Test Split
A single split on `X` is created and then reused (via the shared index) for every target so that every model family sees the exact same rows in train/test.

In [ ]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

## 5. Missing Value Handling
Domain-specific cleanup carried over/expanded from the original notebook. The dataset encodes several kinds of \"missing\" as the sentinel value `-99999`; the correct treatment depends on what the field means.

In [ ]:
# 5a. Drop columns with very high missingness
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# 5b. Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = [
    "age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment",
]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)  # use TRAIN median to avoid leakage

# 5c. Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
]
delinquency_columns = [c for c in delinquency_columns if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# 5d. Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
]
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# 5e. time_since_recent_enq: -99999 = "no enquiry ever" -> longer than any observed gap,
# keep the "no enquiry" signal as its own binary flag (fit on TRAIN only, applied to both)
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# 5f. max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# 5g. Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

## 6. Preprocessing Pipeline (scale numeric, one-hot encode categorical)

In [ ]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

In [ ]:
def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred)        
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [ ]:
feature_names

## Least-Important Features via Lasso
Lasso's L1 penalty drives the coefficients of uninformative features to exactly zero, which makes it a built-in feature-selection tool: sort by `|coefficient|` and whatever sits near/at zero is what Lasso considers least useful for predicting the target. This is done for the **linear** target (`credit_score`, via `LassoCV`) and for **both logistic** targets (`approved_flag` binary & multiclass, via L1-penalized `LogisticRegressionCV`), since Lasso applies equally to regression and classification.

In [ ]:
def get_lasso_ranked_features(coefs, feature_names, zero_tol=1e-4):
    """Return a DataFrame of features sorted ascending by |coefficient| (least
    important first), with a boolean flag for coefficients Lasso zeroed out entirely."""
    pass

### Get least important features for Linear regression and save then to 'lasso_least_important_features_linear.csv'

In [ ]:
# --- Linear regression (credit_score): LassoCV auto-selects alpha via internal CV ---
lasso_cv_linear = LassoCV() # put input parameters
# code to fit and get ranked features
linear_lasso_ranked = pd.DataFrame() # change to save features
linear_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_linear.csv", index=False)
linear_lasso_ranked.head(15)  # 15 LEAST important features

### Get selected features and apply Linear Regression

### Get least important features for Logistic Regression for binary classification and save then to 'lasso_least_important_features_logistic_binary.csv'

In [ ]:
# --- Logistic regression, binary (approved_flag P1/P2 vs P3/P4): L1 LogisticRegressionCV ---
logreg_cv_binary = LogisticRegressionCV() # put input parameters
# code to fit and get ranked features
binary_lasso_ranked = pd.DataFrame() # change to save features
binary_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_logistic_binary.csv", index=False)
binary_lasso_ranked.head(15)

### Get selected features and apply Logistic Regression for Binary Classification

### Get least important features for Logistic Regression for multiclass classification and save then to 'lasso_least_important_features_logistic_multiclass.csv'

In [ ]:
# --- Logistic regression, multiclass (P1/P2/P3/P4): one coefficient vector per class,
# so a feature's overall importance is summarized as the L2 norm across classes ---
logreg_cv_multi = LogisticRegressionCV() # put input parameters
# code to fit and get ranked features
multi_lasso_ranked = pd.DataFrame()
multi_lasso_ranked.to_csv(f"{RESULTS_DIR}/lasso_least_important_features_logistic_multiclass.csv", index=False)
multi_lasso_ranked.head(15)

### Get selected features and apply Logistic Regression for Multiclass Classification

In [ ]:
# --- Combined view: features Lasso considers unimportant across all three targets ---
common_unimportant = set(linear_lasso_ranked.loc[linear_lasso_ranked["zeroed_out"], "feature"]) \
    & set(binary_lasso_ranked.loc[binary_lasso_ranked["zeroed_out"], "feature"]) \
    & set(multi_lasso_ranked.loc[multi_lasso_ranked["zeroed_out"], "feature"])

print(f"{len(common_unimportant)} features zeroed out by Lasso in ALL THREE models "
      f"(safe candidates to drop):")
sorted(common_unimportant)